In [1]:
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
df = pd.read_csv("../data/processed/operations_cleaned.csv")

In [2]:
# ---------------------------------------------------------
# 1. Load data and basic preprocessing
# ---------------------------------------------------------

# Assume you already have a DataFrame `df` similar to your snapshot.
# If reading from CSV:


# Ensure 'date' is a proper datetime and set as index
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['site_id', 'date'])  # site_id is your site column (e.g. 'ite_id' or similar)
df = df.set_index('date')

# If your column is named 'ite_id' in the file, rename for clarity:
df = df.rename(columns={'ite_id': 'site_id'})

In [3]:
df

,site_id,region,behavior,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity,capacity_violation,opening_overflow_tonnes,closing_overflow_tonnes
date,,,,,,,,,,,,,,,
2022-01-01,SITE_001,North,aggressive,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448,False,0.0,0.0
2022-01-02,SITE_001,North,aggressive,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448,False,0.0,0.0
2022-01-03,SITE_001,North,aggressive,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448,False,0.0,0.0
2022-01-04,SITE_001,North,aggressive,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448,False,0.0,0.0
2022-01-05,SITE_001,North,aggressive,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448,False,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-27,SITE_030,South,aggressive,CEM_III,58.48,29.63,0.00,29.63,0.00,7.17,12.00,316,False,0.0,0.0
2024-12-28,SITE_030,South,aggressive,CEM_I,45.39,22.35,0.00,22.35,0.00,2.13,8.04,316,False,0.0,0.0
2024-12-29,SITE_030,South,aggressive,CEM_III,58.47,14.21,0.00,14.21,0.00,0.28,2.26,316,False,0.0,0.0


In [4]:
# ---------------------------------------------------------
# 2. (Optional) Aggregate to weekly per site
#    If you want weekly demand instead of daily.
# ---------------------------------------------------------

# Here we aggregate consumed_tonnes and planned_pour_tonnes by sum per week per site.
# Weather can be mean; behavior/region/cement_type are categorical and carried forward.
weekly = (
    df
    .groupby('site_id')
    .resample('W')  # weekly frequency
    .agg({
        'consumed_tonnes': 'sum',
        'planned_pour_tonnes': 'sum',
        'rain_mm': 'mean',
        'avg_temp_c': 'mean',
        'behavior': 'last',
        'cement_type': 'last',
        'region': 'last',
        'silo_capacity': 'last'
    })
)

# After groupby+resample, 'site_id' becomes index level; reset for easier handling
weekly = weekly.reset_index()

In [5]:
# ---------------------------------------------------------
# 3. Feature engineering: lags and rolling features
# ---------------------------------------------------------

def add_lag_features(df, group_col, target_col, lags, rolling_windows):
    """
    Add lag and rolling features per group (site).
    
    Parameters:
    - df: DataFrame with columns [group_col, target_col, date index or column]
    - group_col: column name identifying the panel (e.g. 'site_id')
    - target_col: column name of the target (e.g. 'consumed_tonnes')
    - lags: list of integers, lag steps (e.g. [1, 2, 4, 8])
    - rolling_windows: list of integers, window sizes for rolling mean (e.g. [4, 8])
    """
    df = df.copy()
    
    # Sort by group and date to ensure correct lagging
    df = df.sort_values([group_col, 'date'])
    
    for lag in lags:
        # Create lag feature per site
        df[f'{target_col}_lag_{lag}'] = (
            df.groupby(group_col)[target_col].shift(lag)
        )
    
    for window in rolling_windows:
        # Rolling mean per site
        df[f'{target_col}_rollmean_{window}'] = (
            df.groupby(group_col)[target_col]
              .shift(1)  # shift by 1 so rolling window doesn't include current value
              .rolling(window=window, min_periods=1)
              .mean()
        )
    
    return df

# Add lag and rolling features to weekly data
weekly = add_lag_features(
    df=weekly,
    group_col='site_id',
    target_col='consumed_tonnes',
    lags=[1, 2, 4, 8],          # lags in weeks
    rolling_windows=[4, 8]     # rolling windows in weeks
)

In [6]:
# ---------------------------------------------------------
# 4. Create multi-step targets (8-week ahead)
# ---------------------------------------------------------

HORIZON = 8  # 8 weeks ahead

def add_future_targets(df, group_col, target_col, horizon):
    """
    Create future target columns for multi-step forecasting.
    
    For each row at time t, we create:
    target_t+1, target_t+2, ..., target_t+horizon
    
    These will be used as multi-output targets in RandomForestRegressor.
    """
    df = df.copy()
    df = df.sort_values([group_col, 'date'])
    
    for h in range(1, horizon + 1):
        df[f'{target_col}_t_plus_{h}'] = (
            df.groupby(group_col)[target_col].shift(-h)
        )
    
    return df

weekly = add_future_targets(
    df=weekly,
    group_col='site_id',
    target_col='consumed_tonnes',
    horizon=HORIZON
)

In [7]:
# ---------------------------------------------------------
# 5. Drop rows with missing values due to lags/targets
# ---------------------------------------------------------

# Rows near the start (lags) and near the end (future targets) will have NaNs.
# We drop them for training.
weekly_model = weekly.dropna().copy()

In [8]:
# ---------------------------------------------------------
# 6. Define features and targets
# ---------------------------------------------------------

# Target columns: 8-week ahead demand
target_cols = [f'consumed_tonnes_t_plus_{h}' for h in range(1, HORIZON + 1)]

# Feature columns: lags, rolling stats, exogenous variables, and site-level info
feature_cols = [
    # lag features
    'consumed_tonnes_lag_1',
    'consumed_tonnes_lag_2',
    'consumed_tonnes_lag_4',
    'consumed_tonnes_lag_8',
    'consumed_tonnes_rollmean_4',
    'consumed_tonnes_rollmean_8',
    
    # exogenous numeric features
    'planned_pour_tonnes',
    'rain_mm',
    'avg_temp_c',
    'silo_capacity',
    
    # categorical features
    'behavior',
    'cement_type',
    'region',
    'site_id'
]

X = weekly_model[feature_cols]
y = weekly_model[target_cols]

In [9]:
# ---------------------------------------------------------
# 7. Build preprocessing + Random Forest pipeline
# ---------------------------------------------------------

# Identify categorical and numeric columns
categorical_cols = ['behavior', 'cement_type', 'region', 'site_id']
numeric_cols = [col for col in feature_cols if col not in categorical_cols]

# ColumnTransformer to one-hot encode categorical variables and pass through numeric
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

# RandomForestRegressor for multi-output regression
rf = RandomForestRegressor(
    n_estimators=300,      # number of trees
    max_depth=None,       # let trees grow; you can tune this
    n_jobs=-1,            # use all cores
    random_state=42
)

# Full pipeline: preprocessing + model
model = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('rf', rf)
])

In [10]:
# ---------------------------------------------------------
# 8. Time-aware train/test split
# ---------------------------------------------------------

# We split by date to respect time ordering.
# For simplicity, use a single cutoff date; you can also use TimeSeriesSplit.

# Choose a cutoff date for training vs testing
cutoff_date = '2022-12-31'  # example; adjust to your data range

train_mask = weekly_model['date'] <= cutoff_date
test_mask = weekly_model['date'] > cutoff_date

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

# Fit the model
model.fit(X_train, y_train)

# Predict on test set
y_pred = model.predict(X_test)

In [11]:
# ---------------------------------------------------------
# 9. Evaluate (simple MAE per horizon)
# ---------------------------------------------------------

# Convert predictions to DataFrame for convenience
y_pred_df = pd.DataFrame(
    y_pred,
    index=y_test.index,
    columns=target_cols
)

# Compute MAE per horizon
mae_per_horizon = {}
for h in range(1, HORIZON + 1):
    col = f'consumed_tonnes_t_plus_{h}'
    mae = mean_absolute_error(y_test[col], y_pred_df[col])
    mae_per_horizon[h] = mae

print("MAE per horizon (weeks ahead):")
for h, mae in mae_per_horizon.items():
    print(f"t+{h}: {mae:.3f}")

MAE per horizon (weeks ahead):
t+1: 26.775
t+2: 27.253
t+3: 27.105
t+4: 27.105
t+5: 27.114
t+6: 27.014
t+7: 26.855
t+8: 28.104


In [12]:
# ---------------------------------------------------------
# 10. Using the model for forecasting future 8-week demand
# ---------------------------------------------------------

def forecast_site_8_weeks(model, weekly_df, site_id, last_date):
    """
    Forecast 8-week demand for a given site starting after `last_date`.
    
    Assumes:
    - weekly_df contains historical data with engineered features.
    - model is the trained pipeline.
    - last_date is the last available date in weekly_df for that site.
    
    Returns:
    - DataFrame with forecasted demand for t+1 ... t+8 weeks.
    """
    # Filter data for the site
    site_data = weekly_df[weekly_df['site_id'] == site_id].copy()
    
    # Ensure sorted
    site_data = site_data.sort_values('date')
    
    # Take the row at last_date (the one we will forecast from)
    current_row = site_data[site_data['date'] == last_date]
    if current_row.empty:
        raise ValueError("No data for given site_id and last_date.")
    
    # We need the same feature columns as training
    X_current = current_row[feature_cols]
    
    # Predict 8-week ahead demand (multi-output)
    y_future = model.predict(X_current)[0]  # shape (8,)
    
    # Build a small DataFrame with future dates and predictions
    future_dates = pd.date_range(
        start=last_date + pd.Timedelta(weeks=1),
        periods=HORIZON,
        freq='W'
    )
    
    forecast_df = pd.DataFrame({
        'site_id': site_id,
        'date': future_dates,
        'forecast_consumed_tonnes': y_future
    })
    
    return forecast_df

# Example usage:
# last_known_date = weekly['date'].max()
# forecast_site_001 = forecast_site_8_weeks(model, weekly_model, 'SITE_001', last_known_date)
# print(forecast_site_001)

In [13]:
# Inspect available sites and dates
print(weekly_model['site_id'].unique())
print(weekly_model.groupby('site_id')['date'].max())


['SITE_001' 'SITE_002' 'SITE_003' 'SITE_004' 'SITE_005' 'SITE_006'
 'SITE_007' 'SITE_008' 'SITE_009' 'SITE_010' 'SITE_011' 'SITE_012'
 'SITE_013' 'SITE_014' 'SITE_015' 'SITE_016' 'SITE_017' 'SITE_018'
 'SITE_019' 'SITE_020' 'SITE_021' 'SITE_022' 'SITE_023' 'SITE_024'
 'SITE_025' 'SITE_026' 'SITE_027' 'SITE_028' 'SITE_029' 'SITE_030']
site_id
SITE_001   2024-11-10
SITE_002   2024-11-10
SITE_003   2024-11-10
SITE_004   2024-11-10
SITE_005   2024-11-10
SITE_006   2024-11-10
SITE_007   2024-11-10
SITE_008   2024-11-10
SITE_009   2024-11-10
SITE_010   2024-11-10
SITE_011   2024-11-10
SITE_012   2024-11-10
SITE_013   2024-11-10
SITE_014   2024-11-10
SITE_015   2024-11-10
SITE_016   2024-11-10
SITE_017   2024-11-10
SITE_018   2024-11-10
SITE_019   2024-11-10
SITE_020   2024-11-10
SITE_021   2024-11-10
SITE_022   2024-11-10
SITE_023   2024-11-10
SITE_024   2024-11-10
SITE_025   2024-11-10
SITE_026   2024-11-10
SITE_027   2024-11-10
SITE_028   2024-11-10
SITE_029   2024-11-10
SITE_030   2024-11

In [14]:
site_id = 'SITE_001'

# Last date for that site in the training dataframe
last_date = weekly_model[weekly_model['site_id'] == site_id]['date'].max()
print(site_id, last_date)


SITE_001 2024-11-10 00:00:00


In [15]:
# Using the helper function defined earlier
forecast_df = forecast_site_8_weeks(
    model=model,              # trained pipeline
    weekly_df=weekly_model,   # dataframe with features+targets (before dropna is also fine if features exist)
    site_id=site_id,
    last_date=last_date
)

print(forecast_df)


    site_id       date  forecast_consumed_tonnes
0  SITE_001 2024-11-17                204.091833
1  SITE_001 2024-11-24                204.572200
2  SITE_001 2024-12-01                200.961500
3  SITE_001 2024-12-08                214.393333
4  SITE_001 2024-12-15                202.770433
5  SITE_001 2024-12-22                204.910667
6  SITE_001 2024-12-29                210.004533
7  SITE_001 2025-01-05                212.700233


In [16]:
def forecast_site_8_weeks(model, weekly_df, site_id, last_date):
    """
    Forecast 8-week demand for a given site starting after `last_date`.
    """
    # Filter data for the chosen site
    site_data = weekly_df[weekly_df['site_id'] == site_id].copy()
    site_data = site_data.sort_values('date')
    
    # Get the row at last_date (this row's features are used to forecast the future)
    current_row = site_data[site_data['date'] == last_date]
    if current_row.empty:
        raise ValueError("No data for given site_id and last_date.")
    
    # Use the same feature columns as during training
    X_current = current_row[feature_cols]
    
    # Predict 8-week ahead demand (multi-output: t+1 ... t+8)
    y_future = model.predict(X_current)[0]  # shape (8,)
    
    # Build future weekly dates
    future_dates = pd.date_range(
        start=last_date + pd.Timedelta(weeks=1),
        periods=HORIZON,
        freq='W'
    )
    
    # Assemble forecast dataframe
    forecast_df = pd.DataFrame({
        'site_id': site_id,
        'date': future_dates,
        'forecast_consumed_tonnes': y_future
    })
    
    return forecast_df


In [19]:
siteID = 'SITE_002'

# Need to get the last_date for SITE_002 specifically
last_date_site_002 = weekly_model[weekly_model['site_id'] == siteID]['date'].max()
print(siteID, last_date_site_002)

# Pass weekly_model (historical data with features), not forecast_df
forecast_site_002 = forecast_site_8_weeks(
    model=model,
    weekly_df=weekly_model,   # historical data with engineered features
    site_id=siteID,
    last_date=last_date_site_002
)

print(forecast_site_002)

SITE_002 2024-11-10 00:00:00
    site_id       date  forecast_consumed_tonnes
0  SITE_002 2024-11-17                 77.705900
1  SITE_002 2024-11-24                 75.523900
2  SITE_002 2024-12-01                 78.871000
3  SITE_002 2024-12-08                 77.705267
4  SITE_002 2024-12-15                 79.894733
5  SITE_002 2024-12-22                 80.844133
6  SITE_002 2024-12-29                 82.459467
7  SITE_002 2025-01-05                 81.656267


In [ ]:
# to check for mutual info score

from sklearn.feature_selection import mutual_info_regression


mi_scores = mutual_info_regression(x, y, random_state=42)
mi_results = pd.DataFrame({
    'Feature': x.columns,
    'MI_Score': mi_scores
}).sort_values(by='MI_Score', ascending=False)

print("\n=== TOP FEATURE MUTUAL INFORMATION RANKINGS ===")
print(mi_results.to_string(index=False))


=== TOP FEATURE MUTUAL INFORMATION RANKINGS ===
                 Feature  MI_Score
     planned_pour_tonnes  4.536621
       deliveries_tonnes  0.864322
closing_inventory_tonnes  0.490850
opening_inventory_tonnes  0.457191
           silo_capacity  0.382425
                 rain_mm  0.064974
              avg_temp_c  0.020303


In [20]:
# ---------------------------------------------------------
# 11. Recursive forecasting beyond the last date in dataset
# ---------------------------------------------------------

def forecast_site_future(model, weekly_df, site_id, start_date, horizon=8, freq='W'):
    """
    Forecast demand for a site starting from `start_date` (can be beyond dataset).
    
    Uses recursive approach: predicts t+1, then uses that prediction to build
    features for t+2, etc.
    """
    site_data = weekly_df[weekly_df['site_id'] == site_id].copy()
    site_data = site_data.sort_values('date')
    
    last_known_row = site_data.iloc[[-1]].copy()
    last_known_date = last_known_row['date'].iloc[0]
    
    # If start_date is within the dataset, use existing function
    if start_date <= last_known_date:
        return forecast_site_8_weeks(model, weekly_df, site_id, start_date)
    
    # Recursive forecasting beyond dataset
    current_features = last_known_row[feature_cols].copy()
    current_date = last_known_date
    forecasts = []
    
    for h in range(1, horizon + 1):
        y_pred = model.predict(current_features)[0]  # shape (8,)
        next_pred = y_pred[0]  # take t+1 prediction
        
        next_date = current_date + pd.Timedelta(weeks=1)
        forecasts.append({
            'site_id': site_id,
            'date': next_date,
            'forecast_consumed_tonnes': next_pred
        })
        
        # Update lag features for next iteration
        new_row = current_features.copy()
        new_row['consumed_tonnes_lag_8'] = new_row['consumed_tonnes_lag_4']
        new_row['consumed_tonnes_lag_4'] = new_row['consumed_tonnes_lag_2']
        new_row['consumed_tonnes_lag_2'] = new_row['consumed_tonnes_lag_1']
        new_row['consumed_tonnes_lag_1'] = next_pred
        
        # Update rolling means (approximate from available lags)
        lag_vals = [
            new_row['consumed_tonnes_lag_1'].iloc[0],
            new_row['consumed_tonnes_lag_2'].iloc[0],
            new_row['consumed_tonnes_lag_4'].iloc[0],
            new_row['consumed_tonnes_lag_8'].iloc[0]
        ]
        lag_vals = [v for v in lag_vals if not pd.isna(v)]
        
        if len(lag_vals) >= 4:
            new_row['consumed_tonnes_rollmean_4'] = np.mean(lag_vals[:4])
            new_row['consumed_tonnes_rollmean_8'] = np.mean(lag_vals)
        elif len(lag_vals) > 0:
            new_row['consumed_tonnes_rollmean_4'] = np.mean(lag_vals)
            new_row['consumed_tonnes_rollmean_8'] = np.mean(lag_vals)
        
        # Exogenous features (planned_pour, weather) held constant at last known
        # Replace with your planned values if available
        
        current_features = new_row
        current_date = next_date
    
    return pd.DataFrame(forecasts)


# Example: Forecast 8 weeks beyond the last date in dataset for SITE_001
site_id = 'SITE_001'
last_date_in_data = weekly_model[weekly_model['site_id'] == site_id]['date'].max()
print(f"Last date in data for {site_id}: {last_date_in_data}")

future_start = last_date_in_data + pd.Timedelta(weeks=1)
print(f"Forecasting from: {future_start}")

future_forecast = forecast_site_future(
    model=model,
    weekly_df=weekly_model,
    site_id=site_id,
    start_date=future_start,
    horizon=8
)

print(future_forecast)

Last date in data for SITE_001: 2024-11-10 00:00:00
Forecasting from: 2024-11-17 00:00:00
    site_id       date  forecast_consumed_tonnes
0  SITE_001 2024-11-17                204.091833
1  SITE_001 2024-11-24                195.370133
2  SITE_001 2024-12-01                199.753700
3  SITE_001 2024-12-08                197.972567
4  SITE_001 2024-12-15                199.606167
5  SITE_001 2024-12-22                195.803167
6  SITE_001 2024-12-29                198.108000
7  SITE_001 2025-01-05                195.968500
